# AAI 510 — Final Project Tool Notebook

## Building Risk Analysis Agent Tools

**Project:** Building Risk  
**Use case:** San Francisco building-level climate and emergency risk analysis  
**Primary dataset:** `carto_overture_maps_buildings.carto.building`

**In this notebook you will:**

- Verify the Overture/CARTO buildings dataset used in the EDA.
- Create a San Francisco building base table with useful geometry-derived fields.
- Create a reusable feature table for proximity-based risk analysis.
- Register three Unity Catalog SQL function tools:
  1. `analyze_urban_fire_spread_risk`
  2. `analyze_emergency_access_bottlenecks`
  3. `rank_buildings_by_composite_risk`
- Test each tool so it is ready to be used by the final AI agent.

### Why these tools?

The EDA showed that the building table has strong building identifiers, bounding boxes, binary geometry, height/floor fields where available, and building class/subtype attributes. These tools start with the features we can support directly from the building dataset, without requiring external FEMA, vegetation, or elevation layers.

---
## 1. Why this agent needs tools

An LLM should not invent building-risk results from general knowledge. For this project, the agent needs tools that can query governed Databricks tables and return grounded risk metrics.

| Tool | What it does | Why it matters |
|---|---|---|
| `analyze_urban_fire_spread_risk` | Finds buildings with many neighboring buildings within a close-distance threshold | Dense building adjacency can increase cascading fire-spread risk |
| `analyze_emergency_access_bottlenecks` | Finds dense clusters where narrow gaps may create emergency access constraints | Emergency response teams need to identify likely access bottlenecks |
| `rank_buildings_by_composite_risk` | Combines fire-spread, access, and building consequence indicators | Business users need a simple ranked list and explanation of main drivers |

These are implemented as **Unity Catalog SQL functions** so an AI agent can discover and call them as structured tools.

---
## 2. Install dependencies

This notebook follows the same tool-building pattern as the course assignment notebook. The main dependency is the Unity Catalog AI client, which can be used later when wiring these functions into an agent.

The SQL tools below use Databricks SQL and standard Spark SQL functions. They intentionally use `bbox`-based distance approximations so the notebook can execute even if binary geometry support is not configured.

In [0]:
# Install the UC AI client for agent tool integration.
# Run this once per cluster/session.
%pip install unitycatalog-ai[databricks]

dbutils.library.restartPython()

---
## 3. Configure catalog, schema, and source table

The default names below match the course workspace pattern. Change these if your team is using a different catalog or schema.

**Important:** The source table comes from the EDA notebook:
`carto_overture_maps_buildings.carto.building`

In [0]:
# Project configuration
CATALOG = "main"
SCHEMA = "default"

SOURCE_BUILDINGS_TABLE = "carto_overture_maps_buildings.carto.building"

BASE_TABLE = f"{CATALOG}.{SCHEMA}.br_sf_buildings_base"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.br_sf_building_risk_features"

print("Source table:", SOURCE_BUILDINGS_TABLE)
print("Base table:  ", BASE_TABLE)
print("Feature table:", FEATURE_TABLE)

---
## 4. Verify the source data

This cell confirms that the CARTO/Overture building table is available and has the fields needed by the tools.

The tools use the following fields:

| Field | Use |
|---|---|
| `id` | Unique building identifier |
| `bbox` | Bounding box used for proximity and density calculations |
| `height` | Optional consequence indicator |
| `num_floors` | Optional consequence indicator |
| `class` | Building category for interpretation |
| `subtype` | Building subtype for interpretation |
| `has_parts` | Optional complexity indicator |
| `version` | Data version tracking |

In [0]:
from pyspark.sql import functions as F

df = spark.table(SOURCE_BUILDINGS_TABLE)

print(f"Row count: {df.count():,}")
print(f"Columns: {df.columns}")
df.printSchema()

display(df.limit(5))

---
## 5. Create a San Francisco building base table

The full Overture building dataset is global. For this project, we filter to a San Francisco bounding box and extract useful numeric fields from `bbox`.

### San Francisco bounding box used

This notebook uses an approximate bounding box:

- Longitude: `-123.0` to `-122.3`
- Latitude: `37.6` to `37.9`

### Why use bounding boxes?

The EDA noted that `geom` is a binary geometry column. Depending on the Databricks runtime and geospatial library setup, binary geometry may require additional conversion tools. To keep the agent tools executable, the first version uses `bbox`-based approximations for:

- centroid longitude/latitude
- approximate footprint area
- approximate distance between building envelopes
- density/proximity scoring

In [0]:
spark.sql(f'''
CREATE OR REPLACE TABLE {BASE_TABLE} AS
SELECT
    id AS building_id,
    height,
    num_floors,
    class AS building_class,
    subtype AS building_subtype,
    has_parts,
    version,
    bbox.xmin AS xmin,
    bbox.xmax AS xmax,
    bbox.ymin AS ymin,
    bbox.ymax AS ymax,

    -- Approximate centroid from the bounding box
    (bbox.xmin + bbox.xmax) / 2.0 AS centroid_lon,
    (bbox.ymin + bbox.ymax) / 2.0 AS centroid_lat,

    -- Approximate width, depth, and footprint area in meters.
    -- 111,320 meters is the approximate length of one degree of latitude.
    -- Longitude meters are adjusted by latitude.
    ABS((bbox.xmax - bbox.xmin) * 111320.0 * COS(RADIANS((bbox.ymin + bbox.ymax) / 2.0))) AS approx_width_m,
    ABS((bbox.ymax - bbox.ymin) * 111320.0) AS approx_depth_m,
    ABS((bbox.xmax - bbox.xmin) * 111320.0 * COS(RADIANS((bbox.ymin + bbox.ymax) / 2.0)))
      * ABS((bbox.ymax - bbox.ymin) * 111320.0) AS approx_footprint_area_m2

FROM {SOURCE_BUILDINGS_TABLE}
WHERE bbox.xmin BETWEEN -123.0 AND -122.3
  AND bbox.xmax BETWEEN -123.0 AND -122.3
  AND bbox.ymin BETWEEN 37.6 AND 37.9
  AND bbox.ymax BETWEEN 37.6 AND 37.9
  AND id IS NOT NULL
  AND bbox IS NOT NULL
''')

base_df = spark.table(BASE_TABLE)
print(f"Created {BASE_TABLE}")
print(f"San Francisco building rows: {base_df.count():,}")
display(base_df.limit(10))

---
## 6. Create proximity and risk feature table

This feature table powers the three agent tools.

### Feature logic

For each building, the notebook counts nearby buildings using approximate bounding-box gaps:

- `nearby_building_count_5m`: number of other buildings within 5 meters  
- `constrained_neighbor_count_6m`: number of other buildings within 6 meters  
- `urban_fire_spread_score`: normalized score based on the 5-meter neighbor count  
- `access_bottleneck_score`: normalized score based on the 6-meter neighbor count  
- `building_consequence_score`: normalized score based on height, floors, and footprint area  
- `composite_risk_score`: weighted score combining proximity and consequence indicators  

### Performance note

The self-join uses bounding-box prefilters to reduce pair comparisons. If the table is too large for your cluster, uncomment the `TABLESAMPLE` or add a neighborhood/H3 preprocessing step in the data pipeline.

In [0]:
spark.sql(f'''
CREATE OR REPLACE TABLE {FEATURE_TABLE} AS
WITH base AS (
    SELECT *
    FROM {BASE_TABLE}
    WHERE approx_footprint_area_m2 > 0
),

candidate_pairs AS (
    SELECT
        a.building_id AS building_id,
        b.building_id AS neighbor_building_id,

        -- Approximate horizontal gap between bounding boxes.
        -- If boxes overlap along an axis, gap on that axis is zero.
        GREATEST(
            GREATEST(a.xmin - b.xmax, b.xmin - a.xmax, 0.0)
            * 111320.0 * COS(RADIANS(a.centroid_lat)),
            0.0
        ) AS gap_x_m,

        GREATEST(
            GREATEST(a.ymin - b.ymax, b.ymin - a.ymax, 0.0)
            * 111320.0,
            0.0
        ) AS gap_y_m

    FROM base a
    JOIN base b
      ON a.building_id <> b.building_id

     -- Prefilter using a small degree buffer around the bbox.
     -- 0.0001 degrees is roughly 8-11 meters in San Francisco.
     AND b.xmin <= a.xmax + 0.0001
     AND b.xmax >= a.xmin - 0.0001
     AND b.ymin <= a.ymax + 0.0001
     AND b.ymax >= a.ymin - 0.0001
),

pair_distances AS (
    SELECT
        building_id,
        neighbor_building_id,
        SQRT(POWER(gap_x_m, 2) + POWER(gap_y_m, 2)) AS approx_gap_m
    FROM candidate_pairs
),

proximity AS (
    SELECT
        building_id,
        COUNT(CASE WHEN approx_gap_m <= 5.0 THEN 1 END) AS nearby_building_count_5m,
        COUNT(CASE WHEN approx_gap_m <= 6.0 THEN 1 END) AS constrained_neighbor_count_6m,
        MIN(approx_gap_m) AS nearest_neighbor_gap_m,
        AVG(CASE WHEN approx_gap_m <= 6.0 THEN approx_gap_m END) AS avg_constrained_gap_m
    FROM pair_distances
    GROUP BY building_id
),

scored AS (
    SELECT
        b.*,
        COALESCE(p.nearby_building_count_5m, 0) AS nearby_building_count_5m,
        COALESCE(p.constrained_neighbor_count_6m, 0) AS constrained_neighbor_count_6m,
        p.nearest_neighbor_gap_m,
        p.avg_constrained_gap_m,

        -- Normalize neighbor counts using log scaling.
        LEAST(1.0, LOG1P(COALESCE(p.nearby_building_count_5m, 0)) / LOG(21.0)) AS urban_fire_spread_score,
        LEAST(1.0, LOG1P(COALESCE(p.constrained_neighbor_count_6m, 0)) / LOG(26.0)) AS access_bottleneck_score,

        -- Consequence proxy using footprint, height, and floors where available.
        LEAST(
            1.0,
            0.50 * LEAST(1.0, COALESCE(b.approx_footprint_area_m2, 0.0) / 1000.0)
          + 0.30 * LEAST(1.0, COALESCE(b.height, 0.0) / 30.0)
          + 0.20 * LEAST(1.0, COALESCE(b.num_floors, 0.0) / 10.0)
        ) AS building_consequence_score

    FROM base b
    LEFT JOIN proximity p
      ON b.building_id = p.building_id
)

SELECT
    *,
    ROUND(
        0.45 * urban_fire_spread_score
      + 0.35 * access_bottleneck_score
      + 0.20 * building_consequence_score,
      4
    ) AS composite_risk_score,

    CASE
        WHEN urban_fire_spread_score >= access_bottleneck_score
         AND urban_fire_spread_score >= building_consequence_score
        THEN 'Dense nearby buildings within 5 meters'
        WHEN access_bottleneck_score >= urban_fire_spread_score
         AND access_bottleneck_score >= building_consequence_score
        THEN 'Constrained nearby gaps within 6 meters'
        ELSE 'Large or tall building consequence proxy'
    END AS primary_risk_driver,

    CURRENT_TIMESTAMP() AS feature_created_at

FROM scored
''')

feature_df = spark.table(FEATURE_TABLE)
print(f"Created {FEATURE_TABLE}")
print(f"Feature rows: {feature_df.count():,}")

display(
    feature_df
    .select(
        "building_id",
        "centroid_lon",
        "centroid_lat",
        "nearby_building_count_5m",
        "constrained_neighbor_count_6m",
        "urban_fire_spread_score",
        "access_bottleneck_score",
        "building_consequence_score",
        "composite_risk_score",
        "primary_risk_driver"
    )
    .orderBy(F.desc("composite_risk_score"))
    .limit(10)
)

---
## 7. Create Unity Catalog SQL function tools

The following cells register the three tools as Unity Catalog functions.

The function comments are intentionally descriptive because the agent uses these descriptions to decide when to call each tool.

### 7.1 Tool: `analyze_urban_fire_spread_risk`

This tool identifies buildings that may have elevated urban fire-spread risk because many other buildings are within a close proximity threshold.

The default threshold is 5 meters.

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.analyze_urban_fire_spread_risk(
  distance_threshold_meters INT COMMENT 'Distance threshold in meters. Current feature table supports best interpretation at 5 meters.',
  result_limit INT COMMENT 'Maximum number of buildings to return.'
)
RETURNS TABLE (
  building_id STRING,
  centroid_lon DOUBLE,
  centroid_lat DOUBLE,
  building_class STRING,
  building_subtype STRING,
  height DOUBLE,
  num_floors DOUBLE,
  approx_footprint_area_m2 DOUBLE,
  nearby_building_count_5m BIGINT,
  nearest_neighbor_gap_m DOUBLE,
  urban_fire_spread_score DOUBLE,
  primary_risk_driver STRING,
  interpretation STRING
)
LANGUAGE SQL
COMMENT 'Ranks San Francisco buildings by urban fire spread risk using nearby building density within approximately 5 meters.'
RETURN (
  WITH ranked AS (
    SELECT
      building_id,
      centroid_lon,
      centroid_lat,
      building_class,
      building_subtype,
      height,
      num_floors,
      approx_footprint_area_m2,
      nearby_building_count_5m,
      nearest_neighbor_gap_m,
      ROUND(urban_fire_spread_score, 4) AS urban_fire_spread_score,
      primary_risk_driver,
      CONCAT(
        'This building has ',
        CAST(nearby_building_count_5m AS STRING),
        ' nearby buildings within the close-distance threshold. Higher counts indicate more potential for cascading urban fire spread.'
      ) AS interpretation,
      ROW_NUMBER() OVER (
        ORDER BY urban_fire_spread_score DESC, nearby_building_count_5m DESC, approx_footprint_area_m2 DESC
      ) AS rn
    FROM main.default.br_sf_building_risk_features
    WHERE nearby_building_count_5m > 0
  )
  SELECT
    building_id,
    centroid_lon,
    centroid_lat,
    building_class,
    building_subtype,
    height,
    num_floors,
    approx_footprint_area_m2,
    nearby_building_count_5m,
    nearest_neighbor_gap_m,
    urban_fire_spread_score,
    primary_risk_driver,
    interpretation
  FROM ranked
  WHERE rn <= result_limit
  ORDER BY rn
)

In [0]:
%sql
SELECT *
FROM main.default.analyze_urban_fire_spread_risk(5, 10)

### 7.2 Tool: `analyze_emergency_access_bottlenecks`

This tool identifies dense building areas where narrow gaps between structures may create emergency access challenges.

The default threshold is 6 meters. This is a first-pass screening tool and should be validated later with road width, hydrant, and emergency route data.

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.analyze_emergency_access_bottlenecks(
  gap_threshold_meters INT COMMENT 'Gap threshold in meters. Current feature table supports best interpretation at 6 meters.',
  result_limit INT COMMENT 'Maximum number of buildings to return.'
)
RETURNS TABLE (
  building_id STRING,
  centroid_lon DOUBLE,
  centroid_lat DOUBLE,
  building_class STRING,
  building_subtype STRING,
  height DOUBLE,
  num_floors DOUBLE,
  approx_footprint_area_m2 DOUBLE,
  constrained_neighbor_count_6m BIGINT,
  avg_constrained_gap_m DOUBLE,
  access_bottleneck_score DOUBLE,
  primary_risk_driver STRING,
  interpretation STRING
)
LANGUAGE SQL
COMMENT 'Ranks San Francisco buildings by emergency access bottleneck risk using constrained neighboring-building gaps of approximately 6 meters. Use this when the user asks about narrow gaps, access constraints, or emergency response bottlenecks.'
RETURN (
  WITH ranked AS (
    SELECT
      building_id,
      centroid_lon,
      centroid_lat,
      building_class,
      building_subtype,
      height,
      num_floors,
      approx_footprint_area_m2,
      constrained_neighbor_count_6m,
      avg_constrained_gap_m,
      ROUND(access_bottleneck_score, 4) AS access_bottleneck_score,
      primary_risk_driver,
      CONCAT(
        'This building has ',
        CAST(constrained_neighbor_count_6m AS STRING),
        ' neighboring buildings within the constrained-gap threshold. High values may indicate emergency access bottlenecks that need road-network validation.'
      ) AS interpretation,
      ROW_NUMBER() OVER (
        ORDER BY access_bottleneck_score DESC, constrained_neighbor_count_6m DESC, approx_footprint_area_m2 DESC
      ) AS rn
    FROM main.default.br_sf_building_risk_features
    WHERE constrained_neighbor_count_6m > 0
  )
  SELECT
    building_id,
    centroid_lon,
    centroid_lat,
    building_class,
    building_subtype,
    height,
    num_floors,
    approx_footprint_area_m2,
    constrained_neighbor_count_6m,
    avg_constrained_gap_m,
    access_bottleneck_score,
    primary_risk_driver,
    interpretation
  FROM ranked
  WHERE rn <= result_limit
  ORDER BY rn
)

In [0]:
%sql
SELECT *
FROM main.default.analyze_emergency_access_bottlenecks(6, 10)

### 7.3 Tool: `rank_buildings_by_composite_risk`

This tool gives the agent a simple ranked list of buildings using the first version of the composite risk score.

The composite score currently combines:

- 45% urban fire-spread score
- 35% emergency access bottleneck score
- 20% building consequence score

This can be updated later when flood, wildfire, storm surge, or earthquake-specific hazard layers are added.

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.rank_buildings_by_composite_risk(
  result_limit INT COMMENT 'Maximum number of buildings to return.',
  minimum_score DOUBLE COMMENT 'Minimum composite risk score from 0 to 1.'
)
RETURNS TABLE (
  rank INT,
  building_id STRING,
  centroid_lon DOUBLE,
  centroid_lat DOUBLE,
  building_class STRING,
  building_subtype STRING,
  height DOUBLE,
  num_floors DOUBLE,
  approx_footprint_area_m2 DOUBLE,
  nearby_building_count_5m BIGINT,
  constrained_neighbor_count_6m BIGINT,
  urban_fire_spread_score DOUBLE,
  access_bottleneck_score DOUBLE,
  building_consequence_score DOUBLE,
  composite_risk_score DOUBLE,
  primary_risk_driver STRING,
  interpretation STRING
)
LANGUAGE SQL
COMMENT 'Ranks San Francisco buildings by a composite risk score using urban fire spread, emergency access bottleneck, and building consequence indicators. Use this when the user asks for highest-risk buildings or an overall building risk ranking.'
RETURN (
  WITH ranked AS (
    SELECT
      CAST(ROW_NUMBER() OVER (
        ORDER BY composite_risk_score DESC, nearby_building_count_5m DESC, constrained_neighbor_count_6m DESC
      ) AS INT) AS rank,
      building_id,
      centroid_lon,
      centroid_lat,
      building_class,
      building_subtype,
      height,
      num_floors,
      approx_footprint_area_m2,
      nearby_building_count_5m,
      constrained_neighbor_count_6m,
      ROUND(urban_fire_spread_score, 4) AS urban_fire_spread_score,
      ROUND(access_bottleneck_score, 4) AS access_bottleneck_score,
      ROUND(building_consequence_score, 4) AS building_consequence_score,
      ROUND(composite_risk_score, 4) AS composite_risk_score,
      primary_risk_driver,
      CONCAT(
        'Composite risk is driven primarily by: ',
        primary_risk_driver,
        '. Scores are based on proximity-derived fire spread, emergency access, and consequence proxies from the building dataset.'
      ) AS interpretation
    FROM main.default.br_sf_building_risk_features
    WHERE composite_risk_score >= minimum_score
  )
  SELECT
    rank,
    building_id,
    centroid_lon,
    centroid_lat,
    building_class,
    building_subtype,
    height,
    num_floors,
    approx_footprint_area_m2,
    nearby_building_count_5m,
    constrained_neighbor_count_6m,
    urban_fire_spread_score,
    access_bottleneck_score,
    building_consequence_score,
    composite_risk_score,
    primary_risk_driver,
    interpretation
  FROM ranked
  WHERE rank <= result_limit
  ORDER BY rank
)


In [0]:
%sql
SELECT *
FROM main.default.rank_buildings_by_composite_risk(10, 0.0)

---
## 8. Python wrappers for agent development

These wrappers are optional, but useful when the next notebook defines the agent. The agent can call the UC functions directly, or you can wrap the SQL calls in Python functions like the examples below.

The wrappers return JSON-like Python dictionaries that are easy to pass into an LLM response step.

In [0]:
def analyze_urban_fire_spread_risk(distance_threshold_meters: int = 5, result_limit: int = 10) -> list[dict]:
    """
    Python wrapper for the UC SQL function `main.default.analyze_urban_fire_spread_risk`.

    Args:
        distance_threshold_meters: Close-building threshold in meters.
        result_limit: Maximum number of rows to return.

    Returns:
        A list of dictionaries with ranked fire-spread risk results.
    """
    query = f"""
    SELECT *
    FROM main.default.analyze_urban_fire_spread_risk({int(distance_threshold_meters)}, {int(result_limit)})
    """
    return [row.asDict() for row in spark.sql(query).collect()]


def analyze_emergency_access_bottlenecks(gap_threshold_meters: int = 6, result_limit: int = 10) -> list[dict]:
    """
    Python wrapper for the UC SQL function `main.default.analyze_emergency_access_bottlenecks`.

    Args:
        gap_threshold_meters: Constrained-gap threshold in meters.
        result_limit: Maximum number of rows to return.

    Returns:
        A list of dictionaries with ranked access bottleneck results.
    """
    query = f"""
    SELECT *
    FROM main.default.analyze_emergency_access_bottlenecks({int(gap_threshold_meters)}, {int(result_limit)})
    """
    return [row.asDict() for row in spark.sql(query).collect()]


def rank_buildings_by_composite_risk(result_limit: int = 10, minimum_score: float = 0.0) -> list[dict]:
    """
    Python wrapper for the UC SQL function `main.default.rank_buildings_by_composite_risk`.

    Args:
        result_limit: Maximum number of rows to return.
        minimum_score: Minimum composite risk score from 0 to 1.

    Returns:
        A list of dictionaries with ranked composite risk results.
    """
    query = f"""
    SELECT *
    FROM main.default.rank_buildings_by_composite_risk({int(result_limit)}, {float(minimum_score)})
    """
    return [row.asDict() for row in spark.sql(query).collect()]

In [0]:
# Test Python wrappers
fire_results = analyze_urban_fire_spread_risk(5, 3)
access_results = analyze_emergency_access_bottlenecks(6, 3)
composite_results = rank_buildings_by_composite_risk(3, 0.0)

print("Urban fire spread sample:")
display(spark.createDataFrame(fire_results))

print("Emergency access bottleneck sample:")
display(spark.createDataFrame(access_results))

print("Composite risk ranking sample:")
display(spark.createDataFrame(composite_results))

---
## 9. Limitations

### Current limitations

- These tools use bounding-box proximity approximations, not full polygon geometry.
- `urban_fire_spread_score` does not include building material, fire history, hydrant proximity, or wind.
- `access_bottleneck_score` does not yet include road width, road closures, traffic, or fire-station distance.
- `composite_risk_score` is a first-pass course-project score, not an operational risk model.